# `Site` observatory site class sandbox

Sandbox to develop and test the `Site` observatory site class for DEMONEXT

### Dependencies

The `site.py` code requires the following:
 * pathlib
 * YAML
 * logging
 * astropy: coordinates.SkyCoord, Angle, and EarthLocation, units, and time.Time
 * astroplan: Observer class for observing circumstances (sunset/sunrise, etc.)

### SRO site functions

The SRO site server is 192.168.1.57. we map these drives with the information for the Building 14 roof
control system status and the site weather station.
<pre>
    SRO14-roof -> R:
    SRO-Weather -> W:
</pre>
The mappings are on the DEMONEXT PC file system, so this should only be run on the DEMONEXT PC, or
a windows computer where we have dummy file systems prepared (the code should correctly indicate
files are unavailable if run elsewhere).

In [29]:
import demonext
from demonext import site

## Instantiate the Site class for SRO

Opens the named obs file and prints the contents to the screen by hand.

In [31]:
try:
    sro = site.Site()
except Exception as exp:
    print(f"ERROR: {exp}")


## Try the class functions

### Site Methods

```
sunAlt()
    what is the current altitude of the Sun relative to the horizon
isNight()
    is it night (sun below the horizon)?
isDark() 
    is it dark (sun more than 18-degrees below the horizon)?
isTwilight(twAlt=-12)
    is it twilight (sun -12 to -18-degrees below the horizon)?
getRoof() 
    retrieve info on the building roof status (roof dictionary)
roofOpen()
    is the building's roof open (True) or closed (False)
getWeather()
    get site weather station data (weather dictionary)
siteInfo()
    return site weather and roof info as FITS keywords for headers
```

### conversion functions

```
f2c()
    convert temperature in Fahrenheit to Celsius
mph2ms()
    convert speed in miles/hour to meters/second
knots2ms()
    convert speed in knots to meters/second
```

## Where is the Sun?

In [33]:
# Sun altitude relative to the local horizon

sunPos = sro.sunAlt()
print(f"Sun altitude: {sunPos:.4f} deg")

# Day or Night?

if sro.isNight():
    print("Night: the Sun is below the horizon at SRO")
    if sro.isDark():
        print("       it is dark (sun >18-deg below the horizon)")
    else:
        if sro.isTwilight():
            print("       it is twilight (12 < sun < 18-deg below the horizon)")
        else:
            print("       it is dusk (sun < 12-deg below the horizon)")
else:
    print("  Day: the Sun is above the horizon at SRO")



Sun altitude: 33.2773 deg
  Day: the Sun is above the horizon at SRO


## What is the roof state?

In [35]:
if sro.roofOpen():
    print("The SRO Building 14 roof is OPEN")
else:
    if sro.roof["position"] == "UNKNOWN":
        print(f"SRO Building 14 roof position is UNKNOWN - {sro.msg}")
    else:
        print("The SRO Building 14 roof is CLOSED")

The SRO Building 14 roof is CLOSED


## What is the weather at SRO?

In [37]:
if sro.getWeather():
    if sro.weather['up']:
        print(f"\nSRO Weather at {sro.weather['iso']} PT:")
        print(f"  Air Temp: {sro.weather['airTemp']:.1f} C")
        print(f"  Humidity: {sro.weather['humidity']:.1f}%  dewPoint: {sro.weather['dewpoint']:.1f} C")
        print(f"      Wind: {sro.weather['winds']}, {sro.weather['windspeed']:.2f} m/s")
        print(f"       Sky: {sro.weather['clouds']}, {sro.weather['rain']}, {sro.weather['sky']}, Temp: {sro.weather['skyTemp']:.1f} C")
else:
    print(f"Weather station not readable - {sro.msg}")


SRO Weather at 2026-04-08T16:31:58 PT:
  Air Temp: 17.6 C
  Humidity: 36.0%  dewPoint: 2.5 C
      Wind: calm, 0.00 m/s
       Sky: clear, dry, daylight, Temp: -14.2 C


## Roof and Weather Info in FITS format

In [39]:
sroInfo = sro.siteTelemetry()
print("SRO Site FITS cards:")
for key, value in sroInfo.items():
    print(f"  {key:8s} = '{value}'")

SRO Site FITS cards:
  SRO_ROOF = 'CLOSED'
  SRO_LINK = 'UP'
  SRO_TEMP = '17.6'
  SRO_HUM  = '36.0'
  SRO_DEWP = '2.5'
  SRO_WIND = 'calm, 0.00 m/s'
  SRO_SKY  = 'clear, dry, daylight, -14.2 C'
